# 零、总述

在上个笔记本中，我们了解了分词和词嵌入的基本概念，现在我们要深入研究语言模型的工作原理了，本节主要讨论 Transformer LLM 的主要工作原理，我们将重点关注文本生成模型，以便更深入地理解生成式 LLM,

本节相关的基础知识和概念都在 `{base_dir}/Notes/Seq2Seq.md` 文件中

# 一、使用 Transformer LLM

In [2]:
%%capture
!pip install transformers>=4.41.2 accelerate>=0.31.0

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

接下来，我们创建一个预测 token 的 Pipeline


此处的 Pipeline 对象是一个封装了 **输入预处理、模型推理和输出后推理** 的可调用对象，它更像是一个 任务执行器/模型调用器

In [4]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


让我们使用 Pipeline(generator) 来续写如下文段

In [5]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my sincerest apologies for the unfortunate incident that occurred


我们看一下 `microsoft/Phi-3-mini-4k-instruct` 模型的架构

In [6]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

我们来看一下此模型的层级结构：
```text
Phi3ForCausalLM
├── model: Phi3Model
│   ├── embed_tokens: Embedding
│   ├── layers: ModuleList
│   │   └── 32 × Phi3DecoderLayer
│   │       ├── self_attn: Phi3Attention
│   │       │   ├── o_proj: Linear
│   │       │   └── qkv_proj: Linear
│   │       ├── mlp: Phi3MLP
│   │       │   ├── gate_up_proj: Linear
│   │       │   ├── down_proj: Linear
│   │       │   └── activation_fn: SiLUActivation
│   │       ├── input_layernorm: Phi3RMSNorm
│   │       ├── post_attention_layernorm: Phi3RMSNorm
│   │       ├── resid_attn_dropout: Dropout
│   │       └── resid_mlp_dropout: Dropout
│   ├── norm: Phi3RMSNorm
│   └── rotary_emb: Phi3RotaryEmbedding
└── lm_head: Linear
```


首先, Phi3ForCausalLM 的第一个子层包含两个组件：**model(Phi3Model)** 和 **lm_head**
- `model`: 负责提取每个 Token 的上下文表示
- `lm_head`: 将这些上下文表示映射到整个词表，生成每个 Token 的预测分数

紧接着，第二个子层的链路是
```text
Embedding -> Layers(32 x Decoder Layer) -> Normalize(RMSNorm) -> RoEmbedding
```

每个 Decoder Layer 层的组件搭建顺序是
```text
Self-Attention -> MLP -> Norm -> Residual Attention Dropout -> Residual MLP Dropout
```


In [7]:
prompt = "The capital of France is"

# 将 prompt 进行分词
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

# 获取模型在 lm_head 之前的输出
model_output = model.model(input_ids)

# 获取 lm_head 的输出
lm_head_output = model.lm_head(model_output[0])

In [8]:
lm_head_output.shape

torch.Size([1, 5, 32064])

现在，lm_head_output 的形状是`[1, 5, 32064]`，我们可以使用 `lm_head_output[0, -1]` 来访问最后生成的词元的概率分数，其中索引 `0` 是批次的维度，表示一批数据中的第一个，索引 `-1` 是用于获取序列中的最后一个 token 的概率分布，我们选择概率最大的一个并解码

In [9]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

In [10]:
model_output[0].shape

torch.Size([1, 5, 3072])

# 二、 KV-Cache

在 Hugging Face 的 Transformers 中，KV-Cache 是默认启用的，我们在 jupyter notebook 中可以使用 `%%timeit` 来感受模型是否启用 KV-Cache 的时间差异

In [11]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# 对 prompt 进行分词
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

使用 KV-Cache

In [12]:
%%timeit -n 1
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=True
)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


4.66 s ± 273 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


不使用 KV-Cache

In [13]:
%%timeit -n 1
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=False
)

35.6 s ± 664 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


**可以看到，在同样的条件下，使用 KV-Cache 比不使用快了将近 ⚠️⚠️⚠️10 倍**